# Polynomial Regression - Starter Notebook
Polynomial regression extends linear regression by adding polynomial feature transforms to capture non-linear relationships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
np.random.seed(42)
n = 100
X = np.sort(np.random.uniform(-3, 3, n))[:, None]
y = 0.5 * X.ravel()**3 - 2 * X.ravel() + np.random.randn(n) * 2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_plot = np.linspace(-3, 3, 300)[:, None]

In [ ]:
# Fit models at different degrees
degrees = [1, 2, 3, 5, 9, 15]
results = []

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
axes = axes.flatten()

for ax, d in zip(axes, degrees):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse  = mean_squared_error(y_test,  model.predict(X_test))
    r2        = r2_score(y_test, model.predict(X_test))
    results.append({'degree': d, 'train_mse': train_mse, 'test_mse': test_mse, 'r2': r2})

    ax.scatter(X_train, y_train, color='gray', s=12, alpha=0.6)
    ax.scatter(X_test,  y_test,  color='orange', s=12, alpha=0.6)
    ax.plot(X_plot, model.predict(X_plot), color='steelblue', linewidth=2)
    ax.set_title(f'Degree={d}\nTrain MSE={train_mse:.1f}  Test MSE={test_mse:.1f}', fontsize=8)
    ax.set_ylim(-25, 25)

plt.suptitle('Polynomial Regression: Underfitting → Overfitting', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Bias-variance trade-off curve
plt.figure(figsize=(7, 4))
plt.plot(results_df['degree'], results_df['train_mse'], 'o-', label='Train MSE')
plt.plot(results_df['degree'], results_df['test_mse'],  's--', label='Test MSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE')
plt.title('Polynomial Regression: Bias-Variance Trade-off')
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# CV-based degree selection
cv_means = []
for d in range(1, 16):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    cv_means.append(-scores.mean())

best_d = np.argmin(cv_means) + 1
print(f'Best degree by 5-fold CV: {best_d}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, 16), cv_means, 'o-', color='darkorange')
plt.axvline(best_d, color='red', linestyle='--', label=f'Best degree={best_d}')
plt.xlabel('Degree')
plt.ylabel('CV MSE')
plt.title('Polynomial Regression: Cross-Validation Degree Selection')
plt.legend()
plt.tight_layout()
plt.show()